In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
df = pd.read_csv("uk_supermarket_supply_2000.csv")

print(df.head())

         Date      Region   Store_ID       Category  Opening_Stock  \
0  01/01/2025    Scotland  UK-SCO-79           Meat            162   
1  01/01/2025    Scotland  UK-SCO-39  Fresh Produce            437   
2  01/01/2025    Scotland  UK-SCO-90           Meat            360   
3  01/01/2025  North West  UK-NOR-34           Meat            396   
4  01/01/2025   Yorkshire  UK-YOR-13           Meat             85   

   Deliveries  Daily_Sales  Closing_Stock  Lead_Time_Days Supplier_Status  \
0         175          136            201               5       Cancelled   
1          94           65            466               7         On-Time   
2         139          117            382               6         Delayed   
3          21          359             58               1         On-Time   
4         164          127            122               2         On-Time   

  Stock_Status  
0      Healthy  
1      Healthy  
2      Healthy  
3      Warning  
4      Warning  


In [3]:
# Check null values first
print(df.isnull().sum())

Date               0
Region             0
Store_ID           0
Category           0
Opening_Stock      0
Deliveries         0
Daily_Sales        0
Closing_Stock      0
Lead_Time_Days     0
Supplier_Status    0
Stock_Status       0
dtype: int64


In [4]:
print(df.duplicated().sum())

0


In [5]:
# Convert Date column to datetime format
df['Date'] = pd.to_datetime(
    df['Date'],
    dayfirst=True,
    errors='coerce'
)

In [6]:
# Inventory-related numeric columns
inventory_cols = [
    'Opening_Stock',
    'Deliveries',
    'Daily_Sales',
    'Closing_Stock',
    'Lead_Time_Days'
]

In [7]:
# Convert inventory columns to numeric datatype
for col in inventory_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [8]:
# Dataset information
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             2000 non-null   datetime64[ns]
 1   Region           2000 non-null   object        
 2   Store_ID         2000 non-null   object        
 3   Category         2000 non-null   object        
 4   Opening_Stock    2000 non-null   int64         
 5   Deliveries       2000 non-null   int64         
 6   Daily_Sales      2000 non-null   int64         
 7   Closing_Stock    2000 non-null   int64         
 8   Lead_Time_Days   2000 non-null   int64         
 9   Supplier_Status  2000 non-null   object        
 10  Stock_Status     2000 non-null   object        
dtypes: datetime64[ns](1), int64(5), object(5)
memory usage: 172.0+ KB
None


In [9]:
# Preview transformed dataset
print("\nTransformed Data:")
print(df.head())


Transformed Data:
        Date      Region   Store_ID       Category  Opening_Stock  Deliveries  \
0 2025-01-01    Scotland  UK-SCO-79           Meat            162         175   
1 2025-01-01    Scotland  UK-SCO-39  Fresh Produce            437          94   
2 2025-01-01    Scotland  UK-SCO-90           Meat            360         139   
3 2025-01-01  North West  UK-NOR-34           Meat            396          21   
4 2025-01-01   Yorkshire  UK-YOR-13           Meat             85         164   

   Daily_Sales  Closing_Stock  Lead_Time_Days Supplier_Status Stock_Status  
0          136            201               5       Cancelled      Healthy  
1           65            466               7         On-Time      Healthy  
2          117            382               6         Delayed      Healthy  
3          359             58               1         On-Time      Warning  
4          127            122               2         On-Time      Warning  


In [10]:
anomalies = df[
    df['Daily_Sales'] >
    (df['Opening_Stock'] + df['Deliveries'])
]

In [11]:
print("Anomalous Records:")
print(anomalies)

Anomalous Records:
           Date         Region   Store_ID       Category  Opening_Stock  \
14   2025-01-02       Scotland  UK-SCO-60  Fresh Produce             71   
285  2025-02-21       Scotland  UK-SCO-79         Bakery            116   
305  2025-02-23     North West  UK-NOR-87  Fresh Produce            312   
315  2025-02-24  West Midlands  UK-WES-56        Festive            386   
428  2025-03-19      Yorkshire  UK-YOR-12        Festive            253   
468  2025-03-25       Scotland  UK-SCO-84        Festive            143   
487  2025-03-29     North West  UK-NOR-46         Bakery            184   
512  2025-04-02         London  UK-LON-52           Meat             95   
531  2025-04-05     North West  UK-NOR-16         Bakery            466   
607  2025-04-18      Yorkshire  UK-YOR-97         Frozen            226   
641  2025-04-26  West Midlands  UK-WES-24           Meat            140   
655  2025-04-28         London  UK-LON-95          Dairy            280   
682  2

In [12]:
print("\nTotal Anomalies Found:", anomalies.shape[0])


Total Anomalies Found: 44


In [13]:
anomalies.to_csv("quarantined_data.csv", index=False)

In [14]:
df_clean = df[
    ~(
        df['Daily_Sales'] >
        (
            df['Opening_Stock'] +
            df['Deliveries']
        )
    )
]

print(df_clean.shape)

(1956, 11)


In [16]:
df_clean.to_csv("cleaned_supply_data.csv", index=False)

In [17]:
def calculate_status(stock):

    if stock < 50:
        return "Critical"

    elif stock < 150:
        return "Warning"

    else:
        return "Healthy"

In [19]:
df_clean.loc[:, 'Stock_Status'] = (
    df_clean['Closing_Stock']
    .apply(calculate_status)
)

print(
    df_clean[
        ['Closing_Stock', 'Stock_Status']
    ].head()
)

   Closing_Stock Stock_Status
0            201      Healthy
1            466      Healthy
2            382      Healthy
3             58      Warning
4            122      Warning


In [31]:
def supplier_score(row):

    score = 100

    if row['Supplier_Status'] == 'On-Time':
        score += 10

    elif row['Supplier_Status'] == 'Delayed':
        score -= 30

    elif row['Supplier_Status'] == 'Cancelled':
        score -= 60

    score -= (
        2 * row['Lead_Time_Days']
    )

    return max(score, 0)

In [34]:
df_clean[
    'Supplier_Reliability_Score'
] = df_clean.apply(
    supplier_score,
    axis=1
)

print(
    df_clean[
        [
            'Supplier_Status',
            'Lead_Time_Days',
            'Supplier_Reliability_Score'
        ]
    ].head()
)

  Supplier_Status  Lead_Time_Days  Supplier_Reliability_Score
0       Cancelled               5                          30
1         On-Time               7                          96
2         Delayed               6                          58
3         On-Time               1                         108
4         On-Time               2                         106


C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_5472\2144040239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[


In [35]:
category_stats = (
    df_clean
    .groupby('Category')['Daily_Sales']
    .agg(['mean', 'std'])
    .reset_index()
)

category_stats['CV'] = (
    category_stats['std']
    /
    category_stats['mean']
)

print(category_stats)

        Category        mean         std        CV
0         Bakery  199.240896  132.681922  0.665937
1          Dairy  192.873016  140.315510  0.727502
2        Festive  197.469027  135.865889  0.688036
3  Fresh Produce  193.075529  139.568611  0.722871
4         Frozen  199.963211  135.549572  0.677873
5           Meat  207.060317  137.141703  0.662327


In [36]:
df_clean = df_clean.merge(
    category_stats[
        ['Category', 'CV']
    ],
    on='Category',
    how='left'
)

print(df_clean.head())

        Date      Region   Store_ID       Category  Opening_Stock  Deliveries  \
0 2025-01-01    Scotland  UK-SCO-79           Meat            162         175   
1 2025-01-01    Scotland  UK-SCO-39  Fresh Produce            437          94   
2 2025-01-01    Scotland  UK-SCO-90           Meat            360         139   
3 2025-01-01  North West  UK-NOR-34           Meat            396          21   
4 2025-01-01   Yorkshire  UK-YOR-13           Meat             85         164   

   Daily_Sales  Closing_Stock  Lead_Time_Days Supplier_Status Stock_Status  \
0          136            201               5       Cancelled      Healthy   
1           65            466               7         On-Time      Healthy   
2          117            382               6         Delayed      Healthy   
3          359             58               1         On-Time      Warning   
4          127            122               2         On-Time      Warning   

   Supplier_Reliability_Score        CV  
0 

In [37]:
df_clean.to_csv(
    "feature_engineered_supply_data.csv",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.


In [38]:
import sqlite3

In [39]:
conn = sqlite3.connect(
    "supply_chain.db"
)

print("Database connected successfully.")

Database connected successfully.


In [4]:
import pandas as pd
import mysql.connector

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="UK_Supermarket"
)

# Query
query = """
SELECT *
FROM feature_engineered_supply_data
LIMIT 10;
"""

# Execute query
df = pd.read_sql(query, conn)

# Show output
display(df)

C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_19128\3618612636.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,Date,Region,Store_ID,Category,Opening_Stock,Deliveries,Daily_Sales,Closing_Stock,Lead_Time_Days,Supplier_Status,Stock_Status,Supplier_Reliability_Score,CV
0,2025-01-01,Scotland,UK-SCO-79,Meat,162,175,136,201,5,Cancelled,Healthy,30,0.662327
1,2025-01-01,Scotland,UK-SCO-39,Fresh Produce,437,94,65,466,7,On-Time,Healthy,96,0.722871
2,2025-01-01,Scotland,UK-SCO-90,Meat,360,139,117,382,6,Delayed,Healthy,58,0.662327
3,2025-01-01,North West,UK-NOR-34,Meat,396,21,359,58,1,On-Time,Warning,108,0.662327
4,2025-01-01,Yorkshire,UK-YOR-13,Meat,85,164,127,122,2,On-Time,Warning,106,0.662327
5,2025-01-01,Yorkshire,UK-YOR-39,Fresh Produce,237,182,153,266,5,On-Time,Healthy,100,0.722871
6,2025-01-01,West Midlands,UK-WES-84,Fresh Produce,427,186,522,91,5,Delayed,Warning,60,0.722871
7,2025-01-01,Scotland,UK-SCO-86,Dairy,356,75,324,107,6,On-Time,Warning,98,0.727502
8,2025-01-02,Scotland,UK-SCO-18,Frozen,185,126,54,257,5,Cancelled,Healthy,30,0.677873
9,2025-01-02,Scotland,UK-SCO-50,Meat,499,64,469,94,2,On-Time,Warning,106,0.662327


In [5]:
query1 = """
SELECT
    Store_ID,
    Category,
    
    SUM(Deliveries) AS Total_Deliveries,
    SUM(Daily_Sales) AS Total_Sales,

    ROUND(
        (SUM(Deliveries) / SUM(Daily_Sales)) * 100,
        2
    ) AS Delivery_Coverage_Percentage,

    ROUND(
        (
            SUM(
                CASE
                    WHEN Supplier_Status IN ('Delayed', 'Cancelled')
                    THEN 1
                    ELSE 0
                END
            ) * 100.0
        ) / COUNT(*),
        2
    ) AS Failure_Percentage

FROM feature_engineered_supply_data

GROUP BY Store_ID, Category

HAVING
    SUM(Deliveries) < (0.5 * SUM(Daily_Sales))

    AND

    (
        SUM(
            CASE
                WHEN Supplier_Status IN ('Delayed', 'Cancelled')
                THEN 1
                ELSE 0
            END
        ) * 100.0
    ) / COUNT(*) > 20;
"""

df1 = pd.read_sql(query1, conn)

display(df1)

C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_19128\3138629305.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


,Store_ID,Category,Total_Deliveries,Total_Sales,Delivery_Coverage_Percentage,Failure_Percentage
0,UK-WES-84,Fresh Produce,381.0,1090.0,34.95,66.67
1,UK-NOR-42,Festive,294.0,617.0,47.65,33.33
2,UK-NOR-10,Bakery,121.0,250.0,48.40,100.00
3,UK-YOR-66,Dairy,19.0,115.0,16.52,50.00
4,UK-NOR-84,Bakery,184.0,433.0,42.49,100.00
...,...,...,...,...,...,...
257,UK-WES-15,Fresh Produce,41.0,297.0,13.80,100.00
258,UK-SCO-36,Bakery,175.0,419.0,41.77,100.00
259,UK-NOR-29,Festive,37.0,164.0,22.56,100.00
260,UK-LON-70,Bakery,56.0,165.0,33.94,100.00


In [6]:
query2 = """
SELECT
    Date,
    Store_ID,
    Region,
    Daily_Sales,
    Supplier_Status

FROM feature_engineered_supply_data f1

WHERE
    Daily_Sales >
    (
        SELECT AVG(f2.Daily_Sales)
        FROM feature_engineered_supply_data f2
        WHERE f1.Region = f2.Region
    )

    AND Supplier_Status = 'Cancelled';
"""

df2 = pd.read_sql(query2, conn)

display(df2)

C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_19128\2625820854.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


,Date,Store_ID,Region,Daily_Sales,Supplier_Status
0,2025-01-05,UK-YOR-13,Yorkshire,239,Cancelled
1,2025-01-08,UK-YOR-72,Yorkshire,398,Cancelled
2,2025-01-11,UK-SCO-73,Scotland,264,Cancelled
3,2025-01-14,UK-YOR-43,Yorkshire,217,Cancelled
4,2025-01-14,UK-NOR-13,North West,286,Cancelled
...,...,...,...,...,...
80,2025-11-28,UK-WES-94,West Midlands,401,Cancelled
81,2025-12-03,UK-NOR-38,North West,351,Cancelled
82,2025-12-04,UK-NOR-74,North West,300,Cancelled
83,2025-12-24,UK-LON-87,London,297,Cancelled


In [7]:
query3 = """
SELECT
    Region,

    ROUND(
        AVG(
            CASE
                WHEN Supplier_Status = 'On-Time'
                THEN Closing_Stock
            END
        ),
        2
    ) AS Avg_Stock_OnTime,

    ROUND(
        AVG(
            CASE
                WHEN Supplier_Status = 'Delayed'
                THEN Closing_Stock
            END
        ),
        2
    ) AS Avg_Stock_Delayed

FROM feature_engineered_supply_data

GROUP BY Region

HAVING
(
    AVG(
        CASE
            WHEN Supplier_Status = 'On-Time'
            THEN Closing_Stock
        END
    )

    -

    AVG(
        CASE
            WHEN Supplier_Status = 'Delayed'
            THEN Closing_Stock
        END
    )
) > 20;
"""

df3 = pd.read_sql(query3, conn)

display(df3)

C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_19128\1250705348.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


,Region,Avg_Stock_OnTime,Avg_Stock_Delayed
0,Scotland,194.47,156.70
1,Yorkshire,190.93,156.78


In [8]:
query4 = """
SELECT
    Store_ID,

    COUNT(*) AS Total_Critical_Flags,

    SUM(
        CASE
            WHEN Closing_Stock >= 150
            THEN 1
            ELSE 0
        END
    ) AS False_Alarms,

    ROUND(
        (
            SUM(
                CASE
                    WHEN Closing_Stock >= 150
                    THEN 1
                    ELSE 0
                END
            ) * 100.0
        ) / COUNT(*),
        2
    ) AS False_Alarm_Percentage

FROM feature_engineered_supply_data

WHERE Stock_Status = 'Critical'

GROUP BY Store_ID

HAVING False_Alarm_Percentage > 0;
"""

df4 = pd.read_sql(query4, conn)

display(df4)

C:\Users\Jaswanth\AppData\Local\Temp\ipykernel_19128\641095482.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql(query4, conn)


,Store_ID,Total_Critical_Flags,False_Alarms,False_Alarm_Percentage
